# Bronze Layer - Zone Reference Data

## Objective

Load NYC Taxi zone reference data and persist it as a Bronze Delta table.

### Input

- taxi_zone_lookup.csv

### Output

- workspace.default.bronze_zones

### Purpose

This dataset will be used to enrich trip records with zone and borough information in the Silver layer.

In [0]:
# Read zone lookup file

zone_df = (
    spark.read
         .option("header", "true")
         .csv(
             "/Volumes/workspace/default/raw_data/taxi_zone_lookup.csv"
         )
)

## Preview Dataset

Review the reference data available for location enrichment.

In [0]:
display(zone_df)

## Schema Review

Inspect column names and datatypes before ingestion.

In [0]:
zone_df.printSchema()

## Add Ingestion Metadata

Capture source and load information for lineage and auditing.

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    lit
)

bronze_zone_df = (
    zone_df
    .withColumn(
        "source_file",
        lit("taxi_zone_lookup.csv")
    )
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
)

## Validate Metadata

Verify metadata columns before persisting the table.

In [0]:
display(
    bronze_zone_df.select(
        "LocationID",
        "Zone",
        "source_file",
        "ingestion_timestamp"
    )
)

## Create Bronze Delta Table

Store the reference dataset in Delta format for downstream processing.

In [0]:
bronze_zone_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.bronze_zones"
    )

## Validation

Confirm all zone records were loaded successfully.

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.default.bronze_zones

## Summary

| Metric | Value |
|----------|----------|
| Output Table | bronze_zones |
| Records Loaded | 265 |

### Metadata Added

- source_file
- ingestion_timestamp

### Next Step

Use the zone lookup table to enrich trip records in the Silver layer.

✅ Bronze Zone Ingestion Completed